# Bankruptcy Prediction

## Data loader

In [1]:
# Library Imports
import pandas as pd
import re

In [9]:
# Calling the data loader and preprocessing script to load the data and perform necessary preprocessing steps for bankruptcy prediction analysis.
%run data_loader_panel.py

Loaded: df_active (Active_companies.xlsx)

  df_active
  Rows: 9,100    Columns: 18

|   # | Column                            | Dtype   |   Non-Null |   Null |
|-----|-----------------------------------|---------|------------|--------|
|   0 | Company name Latin alphabet       | str     |      9,100 |      0 |
|   1 | NACE Rev. 2, core code (4 digits) | str     |      9,100 |      0 |
|   2 | P/L before tax EUR 2024           | Float64 |      8,719 |    381 |
|   3 | P/L before tax EUR 2023           | Float64 |      8,629 |    471 |
|   4 | P/L before tax EUR 2022           | Float64 |      8,315 |    785 |
|   5 | Total assets EUR 2024             | Float64 |      8,721 |    379 |
|   6 | Total assets EUR 2023             | Float64 |      8,633 |    467 |
|   7 | Total assets EUR 2022             | Float64 |      8,318 |    782 |
|   8 | Current liabilities EUR 2024      | Float64 |      8,715 |    385 |
|   9 | Current liabilities EUR 2023      | Float64 |      8,628 |    472 |
|  

## Data Preprocessing 

In [13]:
# ============================================================
# 0. RESET — ensure idempotent execution on re-runs
# ============================================================
# We always work from clean copies of the raw imports.
# This means the block can be re-run any number of times
# without side effects — each run produces identical output.

df_active   = df_active_raw.copy()
df_bankrupt = df_bankrupt_raw.copy()


# ============================================================
# 1. MERGE ACTIVE AND BANKRUPT COMPANIES
# ============================================================
# Assign binary bankruptcy label:
#   0 = active / solvent company
#   1 = bankrupt / dissolved (bankruptcy) company
# Then concatenate into one unified DataFrame.
# ignore_index=True resets row index to 0..N continuously.

df_active["Bankrupt"]   = 0
df_bankrupt["Bankrupt"] = 1

df_analysis = pd.concat([df_active, df_bankrupt], ignore_index=True)

print(f"After concat: {len(df_analysis):,} rows x {len(df_analysis.columns)} columns")
print(f"  Active:   {(df_analysis['Bankrupt'] == 0).sum():,}")
print(f"  Bankrupt: {(df_analysis['Bankrupt'] == 1).sum():,}")


# ============================================================
# 2. MAP NACE CODES TO INDUSTRY SECTIONS
# ============================================================
# Orbis exports 4-digit NACE Rev. 2 codes (e.g. 4711 = supermarkets).
# We group these into broader section-level categories using the
# first 2 digits (division level) per the official Eurostat
# NACE Rev. 2 classification structure.
#
# Sections K, O, P, Q, D, T, U were excluded upstream in Orbis
# via section letter filters and should not appear here.
# Returns "Unknown" for any unmapped code for data quality detection.

nace_section_map = {
    range(1,  4):  "A - Agriculture, forestry, fishing",
    range(5,  10): "B - Mining and quarrying",
    range(10, 34): "C - Manufacturing",
    range(36, 40): "E - Water supply, waste management",
    range(41, 44): "F - Construction",
    range(45, 48): "G - Wholesale and retail trade",
    range(49, 54): "H - Transport and storage",
    range(55, 57): "I - Accommodation and food service",
    range(58, 64): "J - Information and communication",
    range(68, 69): "L - Real estate",
    range(69, 76): "M - Professional, scientific, technical",
    range(77, 83): "N - Administrative and support services",
    range(90, 94): "R - Arts, entertainment, recreation",
    range(94, 97): "S - Other service activities",
}

def nace_to_section(code):
    """
    Maps a 4-digit NACE Rev. 2 code to its section label.
    Uses first 2 digits (division) for lookup.
    Returns 'Unknown' for unmatched codes.
    """
    try:
        division = int(str(code)[:2])
        for r, section in nace_section_map.items():
            if division in r:
                return section
        return "Unknown"
    except:
        return "Unknown"

df_analysis["NACE Section"] = (
    df_analysis["NACE Rev. 2, core code (4 digits)"]
    .apply(nace_to_section)
)

unknown_nace = df_analysis[df_analysis["NACE Section"] == "Unknown"]
if len(unknown_nace) > 0:
    print(f"\nUnknown NACE codes ({len(unknown_nace)} companies):")
    print(unknown_nace["NACE Rev. 2, core code (4 digits)"].value_counts().head(10))
else:
    print("✓ All NACE codes mapped successfully")


# ============================================================
# 3. CLEAN YEAR OF INCORPORATION
# ============================================================
# Force to numeric to handle any non-numeric Orbis export values
# (empty strings, "n.a." etc.) → converted to NaN safely.
# Int64 (nullable integer) used instead of int64 to support NaN.
# Used later to compute company age at each observation year.

df_analysis["Year of incorporation"] = pd.to_numeric(
    df_analysis["Year of incorporation"], errors="coerce"
).astype("Int64")


# ============================================================
# 4. REORDER COLUMNS — IDENTIFIERS FIRST
# ============================================================
# Place identifier and label columns at the front for readability.
# Financial time-series columns follow after.
# Cosmetic only — does not affect downstream modeling.

front_cols = [
    "Company name Latin alphabet",
    "Year of incorporation",
    "Bankrupt",
    "NACE Section",
    "NACE Rev. 2, core code (4 digits)",
]

remaining_cols = [c for c in df_analysis.columns if c not in front_cols]
df_analysis = df_analysis[front_cols + remaining_cols]


# ============================================================
# 5. RESHAPE WIDE → PANEL (LONG) FORMAT
# ============================================================
# Orbis exports in wide format: one row per company, with year-
# suffixed columns for each financial variable:
#   "Total assets EUR 2022", "Total assets EUR 2023", ...
#
# Panel (long) format is required for time-series feature
# engineering: one row per company-year combination, one column
# per financial variable.
#
# Reshape approach:
#   Step A — melt:   wide → long (one row per company × column)
#   Step B — parse:  extract year and metric name from column name
#   Step C — pivot:  long → panel (one row per company × year)

id_cols = [
    "Company name Latin alphabet",
    "Year of incorporation",
    "Bankrupt",
    "NACE Section",
    "NACE Rev. 2, core code (4 digits)",
]

fin_cols = [c for c in df_analysis.columns if c not in id_cols]

# ── Step A: Melt wide → long ─────────────────────────────────
# Result: one row per (company × financial column)
# col_raw = original column name e.g. "Total assets EUR 2022"
# value   = financial figure in thousands EUR
df_long = df_analysis.melt(
    id_vars=id_cols,
    value_vars=fin_cols,
    var_name="col_raw",
    value_name="value",
)

# ── Step B: Parse year and metric name ───────────────────────
# Year:   extract trailing 4-digit year from column name
# Metric: strip "EUR YYYY" suffix to get clean variable name
df_long["Year"] = (
    df_long["col_raw"]
    .str.extract(r"(\d{4})$")
    .astype("Int64")
)

df_long["Metric"] = (
    df_long["col_raw"]
    .str.replace(r"\s*EUR\s*\d{4}$", "", regex=True)
    .str.strip()
)

df_long = df_long.drop(columns="col_raw")

# ── Step C: Pivot long → panel ───────────────────────────────
# Result: one row per (company, year)
# Each metric becomes its own column
# aggfunc="first" handles any accidental duplicates
df_panel = (
    df_long
    .pivot_table(
        index=id_cols + ["Year"],
        columns="Metric",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)
df_panel.columns.name = None


# ============================================================
# 6. DROP FULLY EMPTY ROWS
# ============================================================
# After reshaping, rows exist for every company × year pair even
# when Orbis had no financial data for that year (all NaN values).
# Pre-2016 data is nearly entirely missing — these empty rows
# add no information and inflate memory usage unnecessarily.
#
# Strategy: drop rows where ALL financial metric columns are NaN.
# Rows with partial data are retained — XGBoost handles missing
# values natively so partial observations are still useful.

metric_cols = [c for c in df_panel.columns if c not in id_cols + ["Year"]]

before = len(df_panel)
df_panel = df_panel.dropna(subset=metric_cols, how="all").reset_index(drop=True)
after  = len(df_panel)

print(f"\nDropped {before - after:,} fully empty rows")
print(f"df_panel: {len(df_panel):,} rows x {len(df_panel.columns)} columns")
print(f"  Years covered: {df_panel['Year'].min()} – {df_panel['Year'].max()}")
print(f"  Active rows:   {(df_panel['Bankrupt'] == 0).sum():,}")
print(f"  Bankrupt rows: {(df_panel['Bankrupt'] == 1).sum():,}")


# ============================================================
# 7. COMPUTE COMPANY AGE AT OBSERVATION YEAR
# ============================================================
# Company age = observation year − year of incorporation.
# This gives the company's age in years at the time each
# financial statement was filed — not today's age.
#
# Example: company incorporated 1997, observation year 2022
#          → company age = 25 years at time of filing
#
# Using the observation Year column (panel year) rather than
# current year ensures age is correct for each row in the panel.
# A company with data in 2022, 2023, and 2024 will show ages
# 25, 26, and 27 respectively — reflecting its actual age when
# each set of financials was reported.
#
# Int64 used to support NaN for the one company missing
# Year of incorporation in the raw Orbis export.

df_panel["Company Age"] = (
    df_panel["Year"] - df_panel["Year of incorporation"]
).astype("Int64")

# Sanity checks
age_min = df_panel["Company Age"].min()
age_max = df_panel["Company Age"].max()
age_neg = (df_panel["Company Age"] < 0).sum()
age_null = df_panel["Company Age"].isna().sum()

print(f"Company Age range:     {age_min} – {age_max} years")
print(f"Negative ages (error): {age_neg}")   # should be 0
print(f"Missing ages:          {age_null}")  # should be ≤ 1

# Move Company Age next to Year of incorporation for readability
cols = df_panel.columns.tolist()
year_inc_idx = cols.index("Year of incorporation")
cols.remove("Company Age")
cols.insert(year_inc_idx + 1, "Company Age")
df_panel = df_panel[cols]

print_df_structure("df_panel", df_panel)

After concat: 10,236 rows x 19 columns
  Active:   9,100
  Bankrupt: 1,136
✓ All NACE codes mapped successfully

Dropped 0 fully empty rows
df_panel: 27,965 rows x 11 columns
  Years covered: 2022 – 2024
  Active rows:   25,691
  Bankrupt rows: 2,274
Company Age range:     0 – 119 years
Negative ages (error): 0
Missing ages:          0

  df_panel
  Rows: 27,965    Columns: 12

|   # | Column                            | Dtype   |   Non-Null |   Null |
|-----|-----------------------------------|---------|------------|--------|
|   0 | Company name Latin alphabet       | str     |     27,965 |      0 |
|   1 | Year of incorporation             | Int64   |     27,965 |      0 |
|   2 | Company Age                       | Int64   |     27,965 |      0 |
|   3 | Bankrupt                          | int64   |     27,965 |      0 |
|   4 | NACE Section                      | str     |     27,965 |      0 |
|   5 | NACE Rev. 2, core code (4 digits) | str     |     27,965 |      0 |
|   6 | Yea

In [14]:
from IPython.display import display, HTML
import pandas as pd

def display_scrollable(df, height=400, na_rep=""):
    html = df.to_html(classes="scrollable-table", na_rep=na_rep)
    style = f"""
    <style>
      .scrollable-table-wrapper {{ max-height: {height}px; overflow: auto; border:1px solid #ddd; }}
      .scrollable-table {{ border-collapse: collapse; width:100%; }}
    </style>
    """
    display(HTML(style + f'<div class="scrollable-table-wrapper">{html}</div>'))

# Show bankrupt companies only
display_scrollable(
    df_panel[df_panel['Bankrupt'] == 1],
    height=500
)

,Company name Latin alphabet,Year of incorporation,Company Age,Bankrupt,NACE Section,"NACE Rev. 2, core code (4 digits)",Year,Current assets,Current liabilities,P/L before tax,Shareholders funds,Total assets
10,123 INVEST OY,2013,9,1,G - Wholesale and retail trade,4511,2022,589240.0,254612.0,5316.0,109608.0,667447.0
11,123 INVEST OY,2013,10,1,G - Wholesale and retail trade,4511,2023,685000.0,369000.0,14000.0,121000.0,758000.0
12,123 INVEST OY,2013,11,1,G - Wholesale and retail trade,4511,2024,1188000.0,890000.0,8000.0,124000.0,1244000.0
22,13URLINGTON OY,2017,5,1,G - Wholesale and retail trade,4778,2022,133000.0,288000.0,-115000.0,-178000.0,195000.0
32,1496 SAARA AHO OY,2016,6,1,G - Wholesale and retail trade,4711,2022,81000.0,81000.0,-14000.0,-24000.0,81000.0
33,1496 SAARA AHO OY,2016,7,1,G - Wholesale and retail trade,4711,2023,185000.0,152000.0,-6000.0,-29000.0,185000.0
34,1496 SAARA AHO OY,2016,8,1,G - Wholesale and retail trade,4711,2024,164000.0,114000.0,20000.0,-9000.0,164000.0
65,2020 GROUP OY,2020,2,1,F - Construction,4120,2022,247479.0,215188.0,20485.0,26260.0,260966.0
66,2020 GROUP OY,2020,3,1,F - Construction,4120,2023,286733.0,228780.0,26880.0,47298.0,343509.0
70,247 ERISTYS OY,2019,3,1,F - Construction,4329,2022,420778.0,375039.0,-6916.0,4263.0,427305.0
